# Echo Planar Imaging

Qualitative imaging with an EPI readout.

### Imports

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
from mrpro.algorithms.reconstruction import DirectReconstruction
from mrpro.data import KData
from mrpro.data.traj_calculators import KTrajectoryIsmrmrd
from mrpro.operators import NonUniformFastFourierOp

from mrseq.scripts.epi2d_fid import main as create_seq_fid
from mrseq.scripts.epi2d_se import main as create_seq_se
from mrseq.utils import combine_ismrmrd_files
from mrseq.utils import sys_defaults
from mrseq.utils.trajectory import EpiReadout

### Settings
We are going to use a numerical phantom with a matrix size of 128 x 128. 

In [ ]:
image_matrix_size = [128, 128]
flip_angle_degree = 12
n_dummy_excitations = 200

tmp = tempfile.TemporaryDirectory()

### Create the digital phantom

We use the standard Brainweb phantom from [MRzero](https://github.com/MRsources/MRzero-Core), but we choose the B1-field to be constant everywhere.

In [ ]:
phantom = mr0.util.load_phantom(image_matrix_size)
phantom.B1[:] = 1.0

### EPI readouts

Single-shot EPI readouts are a very efficient way to obtain a full image after a single excitation. 
There are different options of how they can be carried out. 

We can have a symmetric readout where we alternate between going from left to right and right to left through k-space.

In [ ]:
epi_readout = EpiReadout(n_readout=12, n_phase_encoding=12, ramp_sampling=False, readout_type='symmetric')
fig = epi_readout.plot_trajectory()

Here we are sampling only during the flat time of the readout gradient. 
We can make the sequence faster by also sampling during the ramp-up and ramp-down part of the readout gradient.
Nevertheless, this means that we don't have a strictly Cartesian trajectory anymore and gridding/nufft is required 
during image reconstruction.

In [ ]:
epi_readout = EpiReadout(n_readout=12, n_phase_encoding=12, ramp_sampling=True, readout_type='symmetric')
fig = epi_readout.plot_trajectory()

We can shorten the echo time by applying partial Fourier along the phase encoding direction.

In [ ]:
epi_readout = EpiReadout(
    n_readout=12, n_phase_encoding=12, ramp_sampling=True, partial_fourier_factor=0.75, readout_type='symmetric'
)
fig = epi_readout.plot_trajectory()

Another approach is to always go back to one side of k-space after each readout such that the readout direction is always from left to right.
This is less efficient but minimized any artifacts due to asymmetry between positive and negative readout gradients. 

In [ ]:
epi_readout = EpiReadout(
    n_readout=12, n_phase_encoding=12, partial_fourier_factor=0.75, ramp_sampling=True, readout_type='flyback'
)
fig = epi_readout.plot_trajectory()

### EPI sequences

To create the EPI FID and SE sequences, we use the previously imported [epi2d_fid script](../src/mrseq/scripts/ei2d_fid.py) and [epi2d_se script](../src/mrseq/scripts/ei2d_se.py).

We are going to simulate different types of EPI sequences and compare the obtained images.

#### EPI FID with symmetric readout without ramp sampling

In [ ]:
sequence, fname_seq = create_seq_fid(
    system=sys_defaults,
    test_report=False,
    timing_check=False,
    show_plots=False,
    te=0.073,
    fov_xy=float(phantom.size.numpy()[0]),
    n_readout=image_matrix_size[0],
    n_phase_encoding=image_matrix_size[0],
    partial_fourier_factor=0.75,
    ramp_sampling=False,
    readout_type='symmetric',
)

mr0_sequence = mr0.Sequence.import_file(str(fname_seq.with_suffix('.seq')))
signal, ktraj_adc = mr0.util.simulate(mr0_sequence, phantom, accuracy=1e0)
fname_mrd = Path(tmp.name) / 'epi_fid_sym_no_ramp.mrd'
mr0.sig_to_mrd(fname_mrd, signal, sequence)
combine_ismrmrd_files(fname_mrd, Path(f'{fname_seq}_header.h5'))

kdata = KData.from_file(str(fname_mrd).replace('.mrd', '_with_traj.mrd'), trajectory=KTrajectoryIsmrmrd())
recon = DirectReconstruction(kdata, csm=None)
idata_fid_sym_no_ramp = recon(kdata).data

#### EPI FID with symmetric readout and ramp sampling

In [ ]:
sequence, fname_seq = create_seq_fid(
    system=sys_defaults,
    test_report=False,
    timing_check=False,
    show_plots=False,
    te=0.073,
    fov_xy=float(phantom.size.numpy()[0]),
    n_readout=image_matrix_size[0],
    n_phase_encoding=image_matrix_size[0],
    partial_fourier_factor=0.75,
    readout_type='symmetric',
)

mr0_sequence = mr0.Sequence.import_file(str(fname_seq.with_suffix('.seq')))
signal, ktraj_adc = mr0.util.simulate(mr0_sequence, phantom, accuracy=1e0)
fname_mrd = Path(tmp.name) / 'epi_fid_sym_ramp.mrd'
mr0.sig_to_mrd(fname_mrd, signal, sequence)
combine_ismrmrd_files(fname_mrd, Path(f'{fname_seq}_header.h5'))

kdata = KData.from_file(str(fname_mrd).replace('.mrd', '_with_traj.mrd'), trajectory=KTrajectoryIsmrmrd())
fourier_op = NonUniformFastFourierOp(
    direction=(-2, -1),
    recon_matrix=kdata.header.recon_matrix[-2:],
    encoding_matrix=kdata.header.encoding_matrix[-2:],
    traj=kdata.traj,
)
idata_fid_sym_ramp = fourier_op.H(kdata.data)[0]

#### EPI FID with flyback readout and ramp sampling

In [ ]:
sequence, fname_seq = create_seq_fid(
    system=sys_defaults,
    test_report=False,
    timing_check=False,
    show_plots=False,
    te=0.073,
    fov_xy=float(phantom.size.numpy()[0]),
    n_readout=image_matrix_size[0],
    n_phase_encoding=image_matrix_size[0],
    partial_fourier_factor=0.75,
    readout_type='flyback',
)

mr0_sequence = mr0.Sequence.import_file(str(fname_seq.with_suffix('.seq')))
signal, ktraj_adc = mr0.util.simulate(mr0_sequence, phantom, accuracy=1e0)
fname_mrd = Path(tmp.name) / 'epi_fid_flyback_ramp.mrd'
mr0.sig_to_mrd(fname_mrd, signal, sequence)
combine_ismrmrd_files(fname_mrd, Path(f'{fname_seq}_header.h5'))

kdata = KData.from_file(str(fname_mrd).replace('.mrd', '_with_traj.mrd'), trajectory=KTrajectoryIsmrmrd())
fourier_op = NonUniformFastFourierOp(
    direction=(-2, -1),
    recon_matrix=kdata.header.recon_matrix[-2:],
    encoding_matrix=kdata.header.encoding_matrix[-2:],
    traj=kdata.traj,
)
idata_fid_flybackramp = fourier_op.H(kdata.data)[0]

#### EPI SE with symmetric readout and ramp sampling

In [ ]:
sequence, fname_seq = create_seq_se(
    system=sys_defaults,
    test_report=False,
    timing_check=False,
    show_plots=False,
    fov_xy=float(phantom.size.numpy()[0]),
    n_readout=image_matrix_size[0],
    n_phase_encoding=image_matrix_size[0],
    partial_fourier_factor=0.75,
    readout_type='symmetric',
)

mr0_sequence = mr0.Sequence.import_file(str(fname_seq.with_suffix('.seq')))
signal, ktraj_adc = mr0.util.simulate(mr0_sequence, phantom, accuracy=1e-6)
fname_mrd = Path(tmp.name) / 'epi_se_sym_ramp.mrd'
mr0.sig_to_mrd(fname_mrd, signal, sequence)
combine_ismrmrd_files(fname_mrd, Path(f'{fname_seq}_header.h5'))

kdata = KData.from_file(str(fname_mrd).replace('.mrd', '_with_traj.mrd'), trajectory=KTrajectoryIsmrmrd())
fourier_op = NonUniformFastFourierOp(
    direction=(-2, -1),
    recon_matrix=kdata.header.recon_matrix[-2:],
    encoding_matrix=kdata.header.encoding_matrix[-2:],
    traj=kdata.traj,
)
idata_se_sym_ramp = fourier_op.H(kdata.data)[0]

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for cax in ax.flatten():
    cax.set_xticks([])
    cax.set_yticks([])

for idx, (idat, label) in enumerate(
    zip(
        [idata_fid_sym_no_ramp, idata_fid_sym_ramp, idata_fid_flybackramp],
        ['symmetric no ramp', 'symmetric ramp', 'flyback ramp'],
        strict=True,
    )
):
    idat = idat.abs().squeeze().numpy()
    idat /= np.sort(idat.flatten())[int(idat.size * 0.99)]
    if idx == 0:
        idat_ref = idat
        relative_error = 0.0
    relative_error += np.sum(np.abs(idat - idat_ref)) / np.sum(np.abs(idat_ref))

    ax[0, idx].imshow(idat, vmin=0, vmax=1, cmap='gray')
    ax[0, idx].set_title(label)
    ax[1, idx].imshow(idat - idat_ref, vmin=-1, vmax=1, cmap='bwr')

relative_error /= 3

print(f'Relative error {relative_error}')
assert relative_error < 0.15